# HapticGen Inference (Colab)

This notebook runs the `3 image -> OpenAI -> HapticGen` validation flow's **Colab-side inference**.

It does the following:
- clones this repo if needed
- installs `HapticGen` dependencies
- loads `HapticGen/HapticGen-Weights`
- reads either a direct `haptic_prompt` or an `openai_response.json`
- generates one wav
- saves a waveform plot and metadata


In [ ]:
# Setup: clone repo and install dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/cindy-77jiayi/thesis_hapticAE.git'
PROJECT_ROOT = Path('/content/thesis_hapticAE')
BRANCH = 'codex/hapticgen-ui-validation'

if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'HapticGen' / 'requirements.txt')], check=True)
print('Project ready at:', PROJECT_ROOT)


In [ ]:
# Imports and helper functions
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from huggingface_hub import snapshot_download

HAPTICGEN_ROOT = PROJECT_ROOT / 'HapticGen'
if str(HAPTICGEN_ROOT) not in sys.path:
    sys.path.insert(0, str(HAPTICGEN_ROOT))

from audiocraft.data.audio import audio_write
from audiocraft.models import AudioGen

DEFAULT_MODEL_ID = 'HapticGen/HapticGen-Weights'

def load_hapticgen_model(model_id: str, device: str | None = None):
    try:
        model = AudioGen.get_pretrained(model_id, device=device)
        return model, model_id
    except Exception:
        snapshot_dir = snapshot_download(repo_id=model_id)
        model = AudioGen.get_pretrained(snapshot_dir, device=device)
        return model, snapshot_dir

def save_waveform_plot(wav_tensor: torch.Tensor, sample_rate: int, output_path: Path) -> Path:
    waveform = wav_tensor.detach().cpu().squeeze().numpy()
    times = [idx / sample_rate for idx in range(len(waveform))]
    fig = plt.figure(figsize=(12, 3))
    ax = fig.add_subplot(111)
    ax.plot(times, waveform, linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_title('HapticGen Output Waveform')
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)
    return output_path


In [ ]:
# Input configuration
# Use EITHER HAPTIC_PROMPT directly OR OPENAI_RESPONSE_JSON with a `haptic_prompt` field.

HAPTIC_PROMPT = 'A short sharp tap followed by a soft fading buzz.'
OPENAI_RESPONSE_JSON = None

MODEL_ID = DEFAULT_MODEL_ID
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DURATION_SECONDS = 5.0
OUTPUT_DIR = Path('/content/outputs/hapticgen_colab')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if OPENAI_RESPONSE_JSON:
    payload = json.loads(Path(OPENAI_RESPONSE_JSON).read_text(encoding='utf-8'))
    HAPTIC_PROMPT = payload.get('haptic_prompt', HAPTIC_PROMPT)

if not HAPTIC_PROMPT:
    raise ValueError('Set HAPTIC_PROMPT or OPENAI_RESPONSE_JSON before running inference.')

print('Device:', DEVICE)
print('Prompt:', HAPTIC_PROMPT)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# Load model
model, loaded_from = load_hapticgen_model(MODEL_ID, device=DEVICE)
model.set_generation_params(duration=DURATION_SECONDS)
print('Loaded model from:', loaded_from)
print('Sample rate:', model.sample_rate)


In [ ]:
# Generate one sample
generated = model.generate([HAPTIC_PROMPT])
wav_tensor = generated[0].detach().cpu()

wav_stem = OUTPUT_DIR / 'generated'
audio_write(
    str(wav_stem),
    wav_tensor,
    model.sample_rate,
    format='wav',
    add_suffix=False,
    normalize=False,
)
wav_file = OUTPUT_DIR / 'generated.wav'
waveform_file = save_waveform_plot(wav_tensor, model.sample_rate, OUTPUT_DIR / 'generated_waveform.png')

metadata = {
    'prompt': HAPTIC_PROMPT,
    'model_id': MODEL_ID,
    'loaded_from': loaded_from,
    'duration': DURATION_SECONDS,
    'sample_rate': model.sample_rate,
    'wav_path': str(wav_file),
    'waveform_image_path': str(waveform_file),
}
(OUTPUT_DIR / 'generation_metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Saved wav:', wav_file)
print('Saved waveform:', waveform_file)
print('Saved metadata:', OUTPUT_DIR / 'generation_metadata.json')


In [ ]:
# Preview generated artifacts
from IPython.display import Audio, display
from PIL import Image

display(Audio(str(OUTPUT_DIR / 'generated.wav')))
display(Image.open(OUTPUT_DIR / 'generated_waveform.png'))
